# 1. 数据加载

导入所有依赖库，读取Kaggle比赛数据集。

In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

In [ ]:

TRAIN_PATH = Path(r"E:\MLW\demo\train.csv")
TEST_PATH  = Path(r"E:\MLW\demo\test.csv")

# 读取原始数据
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"训练集: {train_df.shape}, 测试集: {test_df.shape}")

训练集: (8693, 14), 测试集: (4277, 13)


# 2. 数据预处理

处理缺失值，软规则补全CryoSleep。

In [3]:
# 五项消费列名（后续多处复用）
SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def _split_or_missing(series, sep, n_parts):
    """拆分结构型字符串列；原值缺失时用占位符填充。"""
    placeholder = sep.join(["Missing"] * n_parts)
    return series.fillna(placeholder).astype(str).str.split(sep, expand=True)


def add_structural_features(df):
    """
    从PassengerId、Cabin、Name中提取结构特征：
    PassengerId -> GroupID / GroupPos
    Cabin       -> Deck / CabinNum / Side
    Name        -> LastName
    """
    out = df.copy()
    pid_parts = _split_or_missing(out["PassengerId"], "_", 2)
    out["GroupID"]  = pid_parts[0].astype(str)
    out["GroupPos"] = pd.to_numeric(pid_parts[1], errors="coerce")

    cabin_parts = _split_or_missing(out["Cabin"], "/", 3)
    out["Deck"]     = cabin_parts[0].astype(str)
    out["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    out["Side"]     = cabin_parts[2].astype(str)

    out["Name"]     = out["Name"].fillna("Missing Missing").astype(str)
    out["LastName"] = out["Name"].str.split().str[-1].astype(str)
    out["Age"] = pd.to_numeric(out["Age"], errors="coerce")
    for col in SPEND_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def apply_soft_cryo_imputation(df):
    """
    软规则补全CryoSleep：
    CryoSleep缺失 且 消费总额>0 => 补False
    （有消费记录的人不可能处于冷冻睡眠状态）
    """
    out = df.copy()
    raw_spend_sum = out[SPEND_COLS].fillna(0).sum(axis=1)
    mask = out["CryoSleep"].isna() & (raw_spend_sum > 0)
    out.loc[mask, "CryoSleep"] = False
    # 消费列NaN统一补0（冷冻状态或未消费）
    for col in SPEND_COLS:
        out[col] = out[col].fillna(0)
    return out

# 3. 特征工程

构造聚合特征（TotalSpend、NoSpend、AgeBin），并对剩余缺失值做兜底填充。

In [ ]:
# 年龄分箱边界（与CatBoost/LightGBM主线保持一致，做受控对比）
AGE_BIN_EDGES  = [-1.0, 12.0, 18.0, 25.0, 40.0, 60.0, 200.0]
AGE_BIN_LABELS = ["child", "teen", "young", "adult", "mid", "senior"]


def add_aggregate_features(df):
    """
    TotalSpend：五项消费之和（反映消费能力和冷冻状态）
    NoSpend   ：是否零消费（与CryoSleep强相关的二值特征）
    AgeBin    ：年龄分箱（减少连续年龄的过拟合风险）
    """
    out = df.copy()
    out["TotalSpend"] = out[SPEND_COLS].sum(axis=1)
    out["NoSpend"]    = (out["TotalSpend"] == 0).astype(int)
    out["Age"]        = out["Age"].fillna(out["Age"].median())
    out["AgeBin"] = pd.cut(
        out["Age"], bins=AGE_BIN_EDGES, labels=AGE_BIN_LABELS, include_lowest=True
    ).astype("object")
    return out


def final_fill(df):
    """兜底填补：数值列用中位数，类别列用字符串Missing。"""
    out = df.copy()
    for col in ["Age", "GroupPos", "CabinNum", "TotalSpend", "NoSpend"]:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(out[col].median())
    for col in ["HomePlanet", "CryoSleep", "Destination", "VIP",
                "GroupID", "Deck", "Side", "LastName", "AgeBin"]:
        out[col] = out[col].fillna("Missing").astype(str)
    return out


def prepare_train_test(train_df, test_df):
    """
    做特征工程：
    保证两边的类别值域、分箱边界、缺失填充口径完全一致。
    """
    train_tmp = train_df.copy(); train_tmp["_is_train"] = 1
    test_tmp  = test_df.copy();  test_tmp["_is_train"]  = 0
    test_tmp["Transported"] = float("nan")

    full = pd.concat([train_tmp, test_tmp], axis=0, ignore_index=True)
    full = add_structural_features(full)
    full = apply_soft_cryo_imputation(full)
    full = add_aggregate_features(full)
    full = final_fill(full)

    train_p = full[full["_is_train"] == 1].copy().drop(columns=["_is_train"])
    test_p  = full[full["_is_train"] == 0].copy().drop(columns=["_is_train"])

    # 19个核心特征（与CatBoost/LightGBM主线完全一致，实现受控对比）
    feature_cols = [
        "HomePlanet", "CryoSleep", "Destination", "Age", "VIP",
        "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
        "GroupID", "GroupPos", "Deck", "CabinNum", "Side",
        "LastName", "TotalSpend", "NoSpend", "AgeBin",
    ]
    categorical_cols = [
        "HomePlanet", "CryoSleep", "Destination", "VIP",
        "GroupID", "Deck", "Side", "LastName", "AgeBin",
    ]
    return train_p, test_p, feature_cols, categorical_cols


# 执行特征工程
train_p, test_p, feature_cols, categorical_cols = prepare_train_test(train_df, test_df)
print(f"特征数: {len(feature_cols)}, 类别特征数: {len(categorical_cols)}")

特征数: 19, 类别特征数: 9


# 4. 编码与标准化

逻辑回归不能直接处理字符串类别，需要先做序数编码（OrdinalEncoder）；
同时逻辑回归对特征尺度敏感，需要对全部特征做标准化（StandardScaler）。

- **OrdinalEncoder**：在train+test合并数据上fit，保证类别映射一致，不引入目标泄漏
- **StandardScaler**：在每折训练集上fit，对验证集和测试集只做transform，严格防止泄漏

In [5]:
# 在train+test合并数据上fit OrdinalEncoder
# 类别映射只依赖特征本身，不涉及目标值，不会引入泄漏
X_all_cat = pd.concat(
    [train_p[categorical_cols], test_p[categorical_cols]],
    axis=0, ignore_index=True
)

cat_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
cat_encoder.fit(X_all_cat)

print(f"OrdinalEncoder 已在 {len(X_all_cat)} 行数据上fit")
print(f"类别列: {categorical_cols}")

OrdinalEncoder 已在 12970 行数据上fit
类别列: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'GroupID', 'Deck', 'Side', 'LastName', 'AgeBin']


# 5. 模型配置

逻辑回归超参数（baseline配置，未做调优）。

In [6]:
# 逻辑回归超参数（baseline基线参数）
LR_PARAMS = dict(
    C           = 1.0,      # 正则化强度的倒数，越小正则化越强
    solver      = "lbfgs",  # L-BFGS拟牛顿法，适合中小规模数据集
    max_iter    = 1000,     # 最大迭代次数（确保收敛）
    random_state= 42,
    n_jobs      = -1,
)

# 交叉验证设置
N_SPLITS     = 5    # 5折交叉验证
RANDOM_STATE = 42

print("模型: LogisticRegression（受控对比基线，与CatBoost/LightGBM使用相同特征）")
print(f"CV策略: {N_SPLITS}折 StratifiedKFold")

模型: LogisticRegression（受控对比基线，与CatBoost/LightGBM使用相同特征）
CV策略: 5折 StratifiedKFold


# 6. 训练

定义模型构建和特征编码函数。

In [7]:
def build_lr_model():
    """构建逻辑回归分类器实例（每折重新初始化）。"""
    return LogisticRegression(**LR_PARAMS)


def encode_fold(X_tr, X_va, X_test, categorical_cols, cat_encoder):
    """
    对单折数据做编码+标准化：
    1. OrdinalEncoder transform（已在全量数据fit，无泄漏）
    2. StandardScaler 在X_tr上fit，X_va/X_test只transform（防止泄漏）
    返回numpy数组，供逻辑回归直接使用。
    """
    X_tr   = X_tr.copy()
    X_va   = X_va.copy()
    X_test = X_test.copy()

    # 序数编码：字符串 -> 整数
    X_tr[categorical_cols]   = cat_encoder.transform(X_tr[categorical_cols])
    X_va[categorical_cols]   = cat_encoder.transform(X_va[categorical_cols])
    X_test[categorical_cols] = cat_encoder.transform(X_test[categorical_cols])

    # 标准化：fit on train only，transform all
    scaler = StandardScaler()
    X_tr_arr   = scaler.fit_transform(X_tr.astype(float))
    X_va_arr   = scaler.transform(X_va.astype(float))
    X_test_arr = scaler.transform(X_test.astype(float))

    return X_tr_arr, X_va_arr, X_test_arr

# 7. 交叉验证

5折StratifiedKFold：每折独立编码、训练，汇总OOF预测评估泛化能力。
逻辑回归无early stopping，直接以max_iter控制训练轮数。

In [8]:
def cross_validate_and_predict(train_p, test_p, feature_cols, categorical_cols, cat_encoder):
    """
    5折分层交叉验证：
    - StratifiedKFold：保证每折正负比例与整体一致
    - 每折内部独立fit StandardScaler，严格防止数据泄漏
    - OOF(Out-of-Fold)：每条训练样本只在没见过它的折上预测，
      汇总后的OOF准确率是模型泛化能力的无偏估计
    """
    X      = train_p[feature_cols].copy()
    y      = train_p["Transported"].astype(int).copy()
    X_test = test_p[feature_cols].copy()

    oof_proba        = np.zeros(len(train_p), dtype=float)
    test_proba_folds = []

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx],  y.iloc[va_idx]

        # 编码 + 标准化（每折独立fit scaler）
        X_tr_enc, X_va_enc, X_te_enc = encode_fold(
            X_tr, X_va, X_test, categorical_cols, cat_encoder
        )

        model = build_lr_model()
        model.fit(X_tr_enc, y_tr)

        va_proba  = model.predict_proba(X_va_enc)[:, 1]
        te_proba  = model.predict_proba(X_te_enc)[:, 1]

        fold_acc = accuracy_score(y_va, (va_proba >= 0.5).astype(int))
        oof_proba[va_idx] = va_proba
        test_proba_folds.append(te_proba)
        print(f"Fold {fold}: accuracy={fold_acc:.5f}")

    # 测试集预测 = 5折概率的平均值
    test_proba = np.mean(np.vstack(test_proba_folds), axis=0)
    oof_pred   = (oof_proba >= 0.5).astype(int)
    oof_acc    = accuracy_score(y, oof_pred)

    print(f"\nCV 本地OOF accuracy: {oof_acc:.5f}")
    return y, oof_proba, oof_pred, test_proba


y_true, oof_proba, oof_pred, test_proba = cross_validate_and_predict(
    train_p, test_p, feature_cols, categorical_cols, cat_encoder
)

Fold 1: accuracy=0.79586
Fold 2: accuracy=0.77343


c:\Users\ASUS\.conda\envs\homework\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\ASUS\.conda\envs\homework\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\ASUS\.conda\envs\homework\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\ASUS\.conda\envs\homework\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs

Fold 3: accuracy=0.79758
Fold 4: accuracy=0.79287
Fold 5: accuracy=0.77848

CV 本地OOF accuracy: 0.78765


c:\Users\ASUS\.conda\envs\homework\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


# 8. 预测

将测试集概率用阈值0.5转为二分类标签，整理为提交格式。

In [9]:
# 概率>=0.5 => Transported=True，<0.5 => False
final_pred = (test_proba >= 0.5)

submission = pd.DataFrame({
    "PassengerId": test_p["PassengerId"].values,
    "Transported": final_pred,
})

print(f"预测Transported=True的比例: {final_pred.mean():.3f}")
submission.head()

预测Transported=True的比例: 0.528


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False


# 9. 提交

保存CSV文件，上传至Kaggle。

> Baseline LogisticRegression Kaggle LB = 0.79471
> （受控对比基线：19特征与CatBoost/LightGBM完全一致）

In [10]:
# 保存提交文件
OUTPUT_PATH = "submission_lr_baseline.csv"
submission.to_csv(OUTPUT_PATH, index=False)
print(f"已保存: {OUTPUT_PATH}")
print(submission["Transported"].value_counts())

已保存: submission_lr_baseline.csv
Transported
True     2258
False    2019
Name: count, dtype: int64
